# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Banking_Transaction_Analytics

**Input file:** `02_Banking_Transactions.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("02_Banking_Transactions.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["amount"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of amount.

In [ ]:
t=    customer_id  customer_name  ...    balance transaction_date
0         B0001  Saurabh Verma  ...  252782.46       2025-10-11
1         B0002     Riya Gupta  ...  352297.18       2025-03-31
2         B0003     Riya Gupta  ...  271455.17       2026-05-08
3         B0004   Pooja Sharma  ...  270758.90       2025-03-24
4         B0005      Nitin Roy  ...  409452.61       2026-09-06
..          ...            ...  ...        ...              ...
397       B0398     Riya Gupta  ...   31354.82       2025-10-08
398       B0399   Manish Kumar  ...  297427.05       2026-04-06
399       B0400   Kavita Singh  ...  301521.76       2025-07-05
400       B0016   Kavita Singh  ...  360718.67       2025-11-09
401       B0032      Nitin Roy  ...  320425.75       2026-09-06

[402 rows x 10 columns]
threshold=df["amount"].quantile(.75)
print(df[df["amount"]>threshold].sort_values("amount",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total amount, and average balance for each customer_id.

In [ ]:
print(df.groupby("customer_id",dropna=False).agg(count=("amount","count"),total=("amount","sum"),avg=("balance","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze amount by customer_id and customer_name; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["customer_id","customer_name"],dropna=False).agg(count=("amount","count"),total=("amount","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["amount"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="customer_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("customer_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["customer_id"]=x["customer_id"].fillna("Unknown"); print(x["customer_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["amount"]=x["amount"].fillna(x["amount"].median()); print(x["amount"].isna().sum())

## Q11. String search

Find records where customer_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["customer_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by amount descending and balance ascending.

In [ ]:
print(df.sort_values(["amount","balance"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on amount using its median as the threshold.

In [ ]:
x=df.copy(); med=x["amount"].median(); x["segment"]=np.where(x["amount"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("customer_id")["amount"].transform("mean"); print(x[["customer_id","amount","group_avg"]].head(25))

## Q15. Group share

Calculate each customer_id's share of total amount.

In [ ]:
g=df.groupby("customer_id",dropna=False)["amount"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average amount by customer_id and customer_name.

In [ ]:
print(pd.pivot_table(df,index="customer_id",columns="customer_name",values="amount",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("customer_id")[["amount","balance"]].mean().reset_index(); print(w.melt(id_vars="customer_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where amount is negative or missing.

In [ ]:
print("Invalid:",df["amount"].isna().sum()+(df["amount"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in amount.

In [ ]:
q1=df["amount"].quantile(.25); q3=df["amount"].quantile(.75); iqr=q3-q1; print(df[df["amount"]>q3+1.5*iqr][["amount"]].head(30))

## Q20. Top group

Find the category in customer_id with the highest total amount.

In [ ]:
print(df.groupby("customer_id")["amount"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in customer_id with the lowest average amount.

In [ ]:
print(df.groupby("customer_id")["amount"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of amount for each customer_id.

In [ ]:
print(df.groupby("customer_id")["amount"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify amount relative to its median.

In [ ]:
x=df.copy(); med=x["amount"].median(); x["classification"]=x["amount"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["customer_id"].dropna().unique())}}; x=df.copy(); x["customer_id_code"]=x["customer_id"].map(codes); print(x[["customer_id","customer_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null amount, non-null customer_id, and values above the median amount.

In [ ]:
m=df["amount"].median(); x=df[df["amount"].notna()&df["customer_id"].notna()&(df["amount"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("02_Banking_Transactions.csv",chunksize=50):
    total += chunk["amount"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final customer_id-level report with count, total amount, average balance, and rank by total amount.

In [ ]:
r=df.groupby("customer_id",dropna=False).agg(records=("amount","count"),total=("amount","sum"),average=("balance","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.